# 1.0 — Getting the Wine Quality data

This notebook downloads the UCI **Wine Quality** dataset ([archive.ics.uci.edu/dataset/186](https://archive.ics.uci.edu/dataset/186/wine+quality)) and stores the **white-wine** data as the project's raw dataset, `data/raw/wine_quality_original.csv`. That file is the starting point of every notebook that follows (1.1 → 1.4).

| File | Folder | Produced by |
|---|---|---|
| `wine+quality.zip` | `data/raw` | this notebook (UCI download) |
| `wine_quality_original.csv` | `data/raw` | this notebook |
| `wine_quality_modified.csv` + `wine_quality_modified_metadata.json` | `data/raw` | provided with the course: a deliberately corrupted copy of the original, used in notebook 1.4 |

> **Paths.** All file locations come from `mlops/config.py` (`RAW_DATA_DIR`, `INTERIM_DATA_DIR`, `PROCESSED_DATA_DIR`), so the notebook works no matter which folder Jupyter is started from. Install the project first: `pip install -r requirements.txt` (it installs the `mlops` package in editable mode).

# Setup

In [ ]:
# Loading the libraries
import io
import zipfile
from urllib.request import urlretrieve

import pandas as pd

from mlops.config import RAW_DATA_DIR

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
ZIP_PATH = RAW_DATA_DIR / "wine+quality.zip"
ORIGINAL_PATH = RAW_DATA_DIR / "wine_quality_original.csv"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Downloading the dataset

The zip is committed to the repository, so the download only happens if it is missing. Raw data is **immutable**: we never edit files in `data/raw`, we only add new ones.

In [ ]:
if ZIP_PATH.exists():
    print(f"Already downloaded: {ZIP_PATH.name}")
else:
    urlretrieve(UCI_URL, ZIP_PATH)
    print(f"Downloaded {UCI_URL} -> {ZIP_PATH.name}")

with zipfile.ZipFile(ZIP_PATH) as zf:
    print(zf.namelist())

# Reviewing the metadata

`winequality.names` documents the variables: 11 physicochemical measurements (inputs) and `quality`, a sensory score between 0 and 10 (output).

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.read("winequality.names").decode("utf-8", errors="replace")

print(names)

# Loading the white-wine data

The UCI files use `;` as separator. We read the white-wine file and save it with the standard `,` separator.

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    wine_quality_df = pd.read_csv(io.BytesIO(zf.read("winequality-white.csv")), sep=";")

wine_quality_df.head()

In [ ]:
wine_quality_df.info()

In [ ]:
# Features (inputs) and target (output)
X = wine_quality_df.drop(columns="quality")
y = wine_quality_df["quality"]

print("Features:", list(X.columns))
print("Target values:", sorted(y.unique()))

# Saving the raw dataset

In [ ]:
wine_quality_df.to_csv(ORIGINAL_PATH, index=False)
print(f"Saved: data/raw/{ORIGINAL_PATH.name}  ({len(wine_quality_df):,} rows, {wine_quality_df.shape[1]} columns)")

# Next steps

- `1.1-grh-intro-numpy-pandas.ipynb` and `1.2-grh-descriptive-statistics.ipynb`: first look at the data with NumPy, pandas and descriptive statistics.
- `1.3-grh-wine-eda.ipynb`: full EDA; also removes duplicate rows and saves `data/interim/wine_quality_original_no_duplicates.csv`.
- `1.4-grh-wine-data-cleaning.ipynb`: cleans `data/raw/wine_quality_modified.csv` and saves the results to `data/processed`.